# RAM++ Kaggle SSH + Finetune Setup
**Flow:** SSH vào Kaggle → clone repo → download weights → prepare data → finetune

## 1. Setup SSH via ngrok

In [ ]:
# ===== CONFIG =====
NGROK_TOKEN = "YOUR_NGROK_AUTHTOKEN"  # https://dashboard.ngrok.com/get-started/your-authtoken
SSH_PASSWORD = "kaggle123"             # password để SSH vào
GITHUB_REPO = "https://github.com/YOUR_USERNAME/image_tagging.git"  # repo URL

In [ ]:
# Cài openssh-server + ngrok
!apt-get update -qq && apt-get install -qq -y openssh-server > /dev/null 2>&1
!pip install pyngrok -q

# Set root password
import subprocess
subprocess.run(["chpasswd"], input=f"root:{SSH_PASSWORD}", text=True, check=True)

# Config SSH cho phép root login
!mkdir -p /var/run/sshd
!echo "PermitRootLogin yes" >> /etc/ssh/sshd_config
!echo "PasswordAuthentication yes" >> /etc/ssh/sshd_config
!/usr/sbin/sshd

# Start ngrok tunnel
from pyngrok import ngrok
ngrok.set_auth_token(NGROK_TOKEN)
tunnel = ngrok.connect(22, "tcp")

# Parse connection info
url = tunnel.public_url.replace("tcp://", "")
host, port = url.split(":")

print("=" * 50)
print("SSH READY! Chạy lệnh này trên local terminal:")
print(f"\n  ssh root@{host} -p {port}\n")
print(f"Password: {SSH_PASSWORD}")
print("=" * 50)

## 2. Cài dependencies + Clone repo
*(Chạy qua SSH hoặc cell tiếp theo)*

In [ ]:
import os
os.chdir("/kaggle/working")

# Clone repo
if not os.path.isdir("image_tagging"):
    !git clone {GITHUB_REPO}

os.chdir("image_tagging/recognize-anything")
!pwd

# Cài deps
!pip install -q timm==0.4.12 "transformers>=4.25.1,<=4.55.4" fairscale==0.4.4 \
    pycocoevalcap ruamel.yaml scipy Pillow
!pip install -q git+https://github.com/openai/CLIP.git

## 3. Download pretrained weights

In [ ]:
!pip install -q huggingface_hub
!mkdir -p pretrained

from huggingface_hub import hf_hub_download

# RAM++ Swin-Large
if not os.path.isfile("pretrained/ram_plus_swin_large_14m.pth"):
    hf_hub_download(
        repo_id="xinyu1205/recognize-anything-plus-model",
        filename="ram_plus_swin_large_14m.pth",
        local_dir="pretrained"
    )
    print("Downloaded RAM++ weights")
else:
    print("Weights already exist")

!ls -lh pretrained/

## 4. Prepare data
Chọn 1 trong 3 options bên dưới

In [ ]:
# === Option A: Dùng Kaggle Dataset (add qua Settings → Add Data) ===
# Ví dụ dataset ở /kaggle/input/my-images/

# !ln -sf /kaggle/input/my-images datasets/custom_imgs

# === Option B: Download data từ URL ===

# !mkdir -p datasets/custom
# !wget -P datasets/custom/ https://your-data-url.com/images.zip
# !unzip -q datasets/custom/images.zip -d datasets/custom/imgs

# === Option C: Dùng dataset có sẵn trong repo ===
!ls datasets/

In [ ]:
# Tạo annotation JSON từ folder ảnh + labels CSV
# Format CSV: image_path,tag1|tag2|tag3,caption text

import json

def create_annotations(csv_path, output_json, tag_list_path="ram/data/ram_tag_list.txt"):
    """Tạo annotation JSON cho finetune từ CSV file."""
    with open(tag_list_path, "r") as f:
        tag_list = [line.strip() for line in f.readlines()]
    tag_to_idx = {tag: idx for idx, tag in enumerate(tag_list)}
    print(f"Loaded {len(tag_list)} tags")

    annotations = []
    skipped = 0
    with open(csv_path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(",", 2)
            img_path = parts[0].strip()
            tags_str = parts[1].strip() if len(parts) > 1 else ""
            caption = parts[2].strip() if len(parts) > 2 else ""

            tag_ids = []
            for t in tags_str.split("|"):
                t = t.strip().lower()
                if t in tag_to_idx:
                    tag_ids.append(tag_to_idx[t])

            if not tag_ids:
                skipped += 1
                continue

            annotations.append({
                "image_path": img_path,
                "caption": caption,
                "union_label_id": tag_ids,
                "parse_label_id": tag_ids
            })

    with open(output_json, "w") as f:
        json.dump(annotations, f, indent=2)

    print(f"Created {len(annotations)} annotations, skipped {skipped}")
    return annotations

# Uncomment để chạy:
# create_annotations("datasets/custom/labels.csv", "datasets/custom/train.json")

## 5. Config finetune

In [ ]:
import torch

# Auto-detect GPU memory để chọn batch size
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"
gpu_mem_gb = torch.cuda.get_device_properties(0).total_mem / 1e9 if torch.cuda.is_available() else 0
n_gpus = torch.cuda.device_count()

# Chọn batch size theo VRAM
if gpu_mem_gb >= 40:    batch_size = 26
elif gpu_mem_gb >= 20:  batch_size = 12
elif gpu_mem_gb >= 15:  batch_size = 8
elif gpu_mem_gb >= 10:  batch_size = 4
else:                   batch_size = 2

print(f"GPU: {gpu_name} | VRAM: {gpu_mem_gb:.1f}GB | Count: {n_gpus} | Batch size: {batch_size}")

In [ ]:
# ===== FINETUNE CONFIG =====
# Sửa train_file theo data của bạn

config_yaml = f"""
train_file:
  - 'datasets/custom/train.json'

image_path_root: ""

vit: 'swin_l'
vit_grad_ckpt: True
vit_ckpt_layer: 0

image_size: 384
batch_size: {batch_size}

weight_decay: 0.05
init_lr: 5e-06
min_lr: 0
max_epoch: 5
warmup_steps: 500

class_num: 4585
"""

with open("ram/configs/finetune_custom.yaml", "w") as f:
    f.write(config_yaml)

print("Config saved to ram/configs/finetune_custom.yaml")
print(config_yaml)

## 6. Finetune!

In [ ]:
import torch
n_gpus = torch.cuda.device_count()

!mkdir -p output/finetune_ram_plus

if n_gpus > 1:
    # Multi-GPU
    !python -m torch.distributed.launch \
        --nproc_per_node={n_gpus} \
        finetune.py \
        --config ram/configs/finetune_custom.yaml \
        --model-type ram_plus \
        --checkpoint pretrained/ram_plus_swin_large_14m.pth \
        --output-dir output/finetune_ram_plus
else:
    # Single GPU
    !CUDA_VISIBLE_DEVICES=0 python -m torch.distributed.launch \
        --nproc_per_node=1 \
        finetune.py \
        --config ram/configs/finetune_custom.yaml \
        --model-type ram_plus \
        --checkpoint pretrained/ram_plus_swin_large_14m.pth \
        --output-dir output/finetune_ram_plus

## 7. Check results

In [ ]:
# Xem training log
!cat output/finetune_ram_plus/log.txt

In [ ]:
# List checkpoints
!ls -lh output/finetune_ram_plus/*.pth

## 8. Quick test inference

In [ ]:
import glob
import torch
from PIL import Image
from ram.models import ram_plus
from ram import inference_ram, get_transform

# Load finetuned model (lấy checkpoint cuối)
ckpts = sorted(glob.glob("output/finetune_ram_plus/checkpoint_*.pth"))
if not ckpts:
    print("No checkpoints found!")
else:
    latest_ckpt = ckpts[-1]
    print(f"Loading: {latest_ckpt}")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # Load checkpoint đúng format finetune (có key 'model')
    ckpt_data = torch.load(latest_ckpt, map_location=device)
    model = ram_plus(pretrained="", image_size=384, vit="swin_l")
    model.load_state_dict(ckpt_data["model"], strict=False)
    model = model.to(device).eval()

    transform = get_transform(image_size=384)

    # Test với 1 ảnh
    test_img = "images/demo/demo1.jpg"  # sửa path
    if os.path.isfile(test_img):
        img = transform(Image.open(test_img).convert("RGB")).unsqueeze(0).to(device)
        tags_en, tags_cn = inference_ram(img, model)
        print(f"Tags: {tags_en}")
    else:
        print(f"Test image not found: {test_img}")

## 9. Download checkpoint về local

In [ ]:
# Zip tất cả checkpoints
!tar -czf /kaggle/working/finetune_result.tar.gz output/finetune_ram_plus/
!ls -lh /kaggle/working/finetune_result.tar.gz

# Download:
# - Kaggle UI: click file trong sidebar → Download
# - SSH/SCP từ local: scp -P <PORT> root@<HOST>:/kaggle/working/finetune_result.tar.gz .

## 10. Keep session alive (optional)
Chạy cell này để giữ Kaggle session không tắt khi train lâu qua SSH

In [ ]:
import time
from IPython.display import clear_output

# Giữ session sống - interrupt cell này để dừng
i = 0
while True:
    i += 1
    clear_output(wait=True)
    print(f"Session alive: {i} min | GPU: {torch.cuda.memory_allocated()/1e9:.1f}GB used")
    time.sleep(60)